# RuleFit with H2O

This example fits a rule ensemble directly to observed Titanic survival labels. It is a predictive model, not a surrogate of a separate black box; the reported held-out AUC measures predictive discrimination. Rule coefficients and importance describe this fitted ensemble and are not causal effects on survival. See [the method and checked rule example](How%20RuleFit%20works.md).

Install the repository requirements and a Java runtime supported by H2O (tested with Java 21). Check Java with `java -version` before starting Jupyter. H2O runs a local Java server; this example uses at most 2 CPU threads and a 2 GB Java heap.

The Titanic data is downloaded from H2O's public example dataset. See the [H2O installation guide](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/downloading.html) for platform-specific setup.

In [ ]:
import h2o
from h2o.estimators import H2ORuleFitEstimator

h2o.init(nthreads=2, max_mem_size="2G")
# Keep track of whether this notebook launched the server.
local_server = h2o.connection().local_server


In [ ]:
# Import the titanic dataset and set the column types:
f = "https://s3.amazonaws.com/h2o-public-test-data/smalldata/gbm_test/titanic.csv"
df = h2o.import_file(path=f, col_types={'pclass': "enum", 'survived': "enum"})

# Split the dataset into train and test
train, test = df.split_frame(ratios=[0.8], seed=1)

# Set the predictors and response:
x = ["age", "sibsp", "parch", "fare", "sex", "pclass"]
y = "survived"

In [ ]:
# Build and train the model:
rfit = H2ORuleFitEstimator(max_rule_length=10,
                           max_num_rules=100,
                           seed=1)
rfit.train(training_frame=train, x=x, y=y)

In [ ]:
# Retrieve the rule importance:
rfit.rule_importance()

In [ ]:
# Predict on the test data:
rfit.predict(test)

In [ ]:
# Evaluate on held-out rows, rather than the training data.
performance = rfit.model_performance(test_data=test)
print(f"Test AUC: {performance.auc():.3f}")

In [ ]:
# Release only the local server started by this notebook.
if local_server is not None:
    local_server.shutdown()
